# 02 — Fine-tuning Language Models for a New Domain

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand what **fine-tuning** is: continue training a *pretrained* model on a small task/domain dataset instead of training from scratch
- **Run a complete fine-tuning experiment on real corpora**: pretrain a language model on one real corpus, measure it on a genuinely different real domain, fine-tune, and measure the improvement
- Observe **catastrophic forgetting** — the classic fine-tuning side effect
- Know the industry workflow (Hugging Face `Trainer`) for fine-tuning real LLMs

## 🔗 Where this fits

**Builds on:** Course 10 — Unit 2, lesson 01 (the pretrained character LM) and Course 07 (AIAT 121) — Unit 4, lesson 03, which walked through the Hugging Face fine-tuning recipe without running it; fine-tuning is the same loop as Course 01 (AIAT 111) — Unit 3, lesson 04, continued at a lower learning rate.

**Used later in:** Course 10 — Unit 2, lesson 03, which steers a model by prompt instead of by further training.

## 📊 Data used in this notebook

Two **real** corpora with a genuinely large domain gap:
- **Base corpus** — 20 Newsgroups `sci.space` posts (`fetch_20newsgroups`): free-form English prose written by real people.
- **Target domain** — free-text `title` field of the **Montgomery County 911 call log** (`Course 04/datasets/raw/montgomery_911_calls.csv`): short, templated emergency-dispatch strings.

Using real corpora matters for this lesson specifically: on invented text a model memorises the base corpus to near-zero loss, which makes the "before fine-tuning" number meaningless. On real prose the pretrained model has a real, non-trivial loss — so the before/after comparison and the forgetting effect are measured on honest baselines.

---

## Introduction

Nobody trains a production language model from scratch for every task. The **transfer learning** recipe is: take a model pretrained on a huge general corpus, then **fine-tune** it — continue training briefly, usually at a lower learning rate — on your small domain dataset.

We run the entire recipe end-to-end at classroom scale with a character-level LSTM: *pretrain* on Usenet prose, then *fine-tune* on emergency-dispatch text. Both corpora are small, but they are real and the measurements are real: the same loss, evaluated on the same held-out text, before and after fine-tuning.

---

## 🌍 Why this lesson exists — Bloomberg paid to avoid the trade-off you are about to measure

BloombergGPT (Wu et al., 2023, arXiv 2303.17564) is a **50-billion-parameter** model trained on **363 billion tokens** of Bloomberg's financial data *plus* **345 billion tokens** of general-purpose text — roughly a 51/49 mix. Bloomberg did not fine-tune a general model on finance alone. They deliberately paid to keep half the training budget on general text, because a finance model that has forgotten how to read ordinary English is not useful to a Bloomberg terminal user. That mix is a company buying its way around the exact effect you are about to measure in Part 2.

**What goes wrong without this:** without re-measuring the *original* domain after fine-tuning, you ship a model that is better at the demo and worse at everything the customer also relied on — and you learn about it from the customer.


## Part 1 — Pretrain on the Base Corpus (real Usenet prose)

One important setup detail: the character vocabulary is built over **both** corpora up front. A real LLM's tokenizer is fixed at pretraining time in exactly the same way — fine-tuning never changes the vocabulary.


In [1]:
# WHAT/WHY: pretrain a character-level LSTM language model on a REAL corpus
# (Usenet sci.space posts) — this plays the role of the "pretrained base
# model" that we fine-tune on a different real domain in Part 2.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import pandas as pd
import re
from sklearn.datasets import fetch_20newsgroups

torch.manual_seed(42); np.random.seed(42)

def normalise(s):
    """Lowercase and keep only letters, digits, space, period and comma.
    A small shared character set lets one vocabulary cover both corpora."""
    s = re.sub(r'[^a-z0-9 .,]', ' ', s.lower())
    return re.sub(r'\s+', ' ', s).strip()

# ── REAL base corpus: free-form Usenet prose (sci.space) ──────────────────
news = fetch_20newsgroups(subset='train', categories=['sci.space'],
                          remove=('headers', 'footers', 'quotes'))
base_text = normalise(" ".join(news.data))[:16000]

# ── REAL target-domain corpus: 911 emergency-dispatch call titles ─────────
CALLS = '../../../Course 04/datasets/raw/montgomery_911_calls.csv'
calls = pd.read_csv(CALLS, usecols=['title'], nrows=6000)
domain_text = normalise(' . '.join(calls['title'].astype(str).tolist()))[:8000]

print(f"base corpus   (Usenet sci.space): {len(base_text):,} chars from {len(news.data)} posts")
print(f"  sample: {base_text[:110]!r}")
print(f"domain corpus (911 dispatch):     {len(domain_text):,} chars from {len(calls):,} calls")
print(f"  sample: {domain_text[:110]!r}")

# ── Vocabulary over BOTH corpora (like a fixed LLM tokenizer) ─────────────
chars = sorted(set(base_text + domain_text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
VOCAB = len(chars)
SEQ_LEN = 20
print(f"\nvocabulary: {VOCAB} characters (shared by both corpora)")

def make_dataset(text):
    """Slice a text into (20-char context → next char) training pairs."""
    enc = [c2i[c] for c in text]
    X = [enc[i:i+SEQ_LEN] for i in range(len(enc) - SEQ_LEN - 1)]
    y = [enc[i+SEQ_LEN]   for i in range(len(enc) - SEQ_LEN - 1)]
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X_base, y_base     = make_dataset(base_text)
X_domain, y_domain = make_dataset(domain_text)
print(f"training windows — base: {len(X_base):,} | domain: {len(X_domain):,}")

# ── The language model (same architecture as example 01) ──────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

model   = CharLM()
loss_fn = nn.CrossEntropyLoss()

def avg_loss(X, y, bs=2048):
    """Average next-character cross-entropy over a dataset (lower = better fit).
    Evaluated in batches so a 16k-window corpus fits comfortably in memory."""
    model.eval(); tot = 0.0; n = 0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb, yb = X[i:i+bs], y[i:i+bs]
            tot += loss_fn(model(xb), yb).item() * len(xb); n += len(xb)
    return tot / n

# ── PRETRAINING: 600 mini-batch steps on the real base corpus ─────────────
opt = optim.Adam(model.parameters(), lr=3e-3)
for step in range(600):
    model.train()
    perm = torch.randperm(len(X_base))[:256]
    loss = loss_fn(model(X_base[perm]), y_base[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 150 == 0:
        print(f"pretrain step {step} — loss: {loss.item():.3f}")

print(f"\nAfter pretraining:")
print(f"  loss on base corpus (Usenet prose):  {avg_loss(X_base, y_base):.3f}")
print(f"  loss on target domain (911 titles):  {avg_loss(X_domain, y_domain):.3f}")
print("The gap is the point: the pretrained model does not fit the new domain yet.")


base corpus   (Usenet sci.space): 16,000 chars from 593 posts
  sample: 'any lunar satellite needs fuel to do regular orbit corrections, and when its fuel runs out it will crash withi'
domain corpus (911 dispatch):     8,000 chars from 6,000 calls
  sample: 'ems back pains injury . ems diabetic emergency . fire gas odor leak . ems cardiac emergency . ems dizziness . '

vocabulary: 39 characters (shared by both corpora)
training windows — base: 15,979 | domain: 7,979


pretrain step 0 — loss: 3.656


pretrain step 150 — loss: 2.198


pretrain step 300 — loss: 1.980


pretrain step 450 — loss: 1.551



After pretraining:


  loss on base corpus (Usenet prose):  1.433
  loss on target domain (911 titles):  2.721
The gap is the point: the pretrained model does not fit the new domain yet.


## Part 2 — Fine-tune on the Target Domain and Measure

Fine-tuning = **the same training loop, continued** — on the new data, with a lower learning rate (so the pretrained weights are adjusted, not destroyed). We measure the domain loss before/after, and also re-measure the base-corpus loss to see **catastrophic forgetting**: improving on the new domain degrades the old one.


In [2]:
# WHAT/WHY: fine-tune the pretrained model on the 911 dispatch corpus with a
# lower learning rate, then measure (a) the improvement on the new domain and
# (b) the regression on the old domain (catastrophic forgetting). Generations
# before/after make the change visible.

def generate(seed, steps=110, temperature=0.8):
    # sample continuation characters one at a time (as in example 01)
    model.eval()
    out = list(seed)
    ctx = [c2i.get(c, 0) for c in seed[-SEQ_LEN:]]
    for _ in range(steps):
        with torch.no_grad():
            logits = model(torch.tensor([ctx[-SEQ_LEN:]]))[0] / temperature
        nxt = int(np.random.choice(VOCAB, p=torch.softmax(logits, 0).numpy()))
        out.append(i2c[nxt]); ctx.append(nxt)
    return ''.join(out)

# ── Snapshot metrics + a sample BEFORE fine-tuning ────────────────────────
np.random.seed(0)
loss_domain_before = avg_loss(X_domain, y_domain)
loss_base_before   = avg_loss(X_base, y_base)
sample_before = generate("ems, cardiac emerge")

# ── FINE-TUNING: 400 steps on the domain corpus, lr 10× lower ─────────────
opt_ft = optim.Adam(model.parameters(), lr=3e-4)   # lower lr: adjust, don't destroy
for step in range(400):
    model.train()
    perm = torch.randperm(len(X_domain))[:256]
    loss = loss_fn(model(X_domain[perm]), y_domain[perm])
    opt_ft.zero_grad(); loss.backward(); opt_ft.step()

# ── Measure AFTER fine-tuning ─────────────────────────────────────────────
np.random.seed(0)
loss_domain_after = avg_loss(X_domain, y_domain)
loss_base_after   = avg_loss(X_base, y_base)
sample_after = generate("ems, cardiac emerge")

print("Next-char cross-entropy loss     before FT    after FT")
print(f"  target domain (911 titles)      {loss_domain_before:.3f}        {loss_domain_after:.3f}")
print(f"  base corpus (Usenet prose)      {loss_base_before:.3f}        {loss_base_after:.3f}")
print()
print("Generation, seed 'ems, cardiac emerge':")
print(f"  BEFORE: {sample_before!r}")
print(f"  AFTER:  {sample_after!r}")
print()
print("Two lessons in the numbers: fine-tuning cut the domain loss, and the")
print("base-corpus loss ROSE — catastrophic forgetting. Production fine-tuning")
print("mitigates it by mixing in base data or freezing/limiting updates (LoRA).")


Next-char cross-entropy loss     before FT    after FT
  target domain (911 titles)      2.721        0.421
  base corpus (Usenet prose)      1.433        2.011

Generation, seed 'ems, cardiac emerge':
  BEFORE: 'ems, cardiac emergep. in problem. a stound allor dod, the, eloence launol are debion test, with a decend latu sepale tate the gav'
  AFTER:  'ems, cardiac emergency . ems subsectives . ems medical emergency . ems wandon . ems gems verifater emergency . ems pal omnters . '

Two lessons in the numbers: fine-tuning cut the domain loss, and the
base-corpus loss ROSE — catastrophic forgetting. Production fine-tuning
mitigates it by mixing in base data or freezing/limiting updates (LoRA).


## 💬 Discuss

1. Domain loss fell 85% (2.721 → 0.421) and base-corpus loss rose 40% (1.433 → 2.011). If the customer only ever sends dispatch text, is the forgetting a problem at all? Name the situation in which it suddenly becomes one.
2. Bloomberg paid for a roughly 50/50 finance/general mix rather than fine-tuning on finance alone. Estimate what that decision cost them in compute, then say when you would recommend the cheaper fine-tuning route instead.
3. We lowered the learning rate by 10× so the pretrained weights would be "adjusted, not destroyed". What would you *measure* to choose that factor properly, rather than picking 10 because it is round?


## The Same Recipe on Real LLMs — Hugging Face Workflow (reference)

On real models the loop you just ran is wrapped by the `transformers` library. **Reference only — not executed in this notebook** (it downloads a ~500 MB pretrained GPT-2):

```python
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)

tok   = AutoTokenizer.from_pretrained("distilgpt2")     # fixed vocabulary
model = AutoModelForCausalLM.from_pretrained("distilgpt2")  # pretrained weights

args = TrainingArguments(output_dir="ft-out",
                         num_train_epochs=3,
                         per_device_train_batch_size=8,
                         learning_rate=5e-5)            # note: low lr, as in Part 2

trainer = Trainer(model=model, args=args,
                  train_dataset=your_tokenized_domain_dataset,
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
trainer.train()
```

Every piece maps onto what you ran: pretrained weights = our Part 1 model; `learning_rate=5e-5` = our lowered lr; the dataset = our 911-dispatch corpus. Parameter-efficient variants (**LoRA**, adapters) fine-tune a small fraction of weights to save memory and reduce forgetting.


## ⚠️ Where this breaks

**Fine-tuning trades one distribution for another; there is no free version.** No learning rate gives you the new domain at zero cost to the old one. The only question is how much of the old you are willing to pay, and the only way to know is to measure both — which is why this notebook prints two rows, not one.

**It cannot add what the base model has no representation for.** Our character vocabulary was frozen at pretraining, exactly as a real tokenizer is. A target domain full of out-of-vocabulary symbols — a new script, chemical formulae, a code language the tokenizer splits badly — is not fixed by fine-tuning.

**Try the cheaper things first.** If your domain fits in the prompt, prompt it (example 03). If you need *facts*, retrieve them rather than training them in. Fine-tune when you need a style, format or behaviour the base model cannot be talked into — and note that our forgetting appeared after a few hundred steps on 8,000 characters.

**LoRA and adapters reduce cost and forgetting; they do not remove either.** They constrain how far the weights can move, which is the same trade-off with a smaller dial — useful, and still a trade-off you must measure.


## 📚 References & Further Reading

**Papers:**
- Howard & Ruder (2018) — [ULMFiT: Universal Language Model Fine-tuning](https://arxiv.org/abs/1801.06146) *(established the pretrain→fine-tune recipe for NLP)*
- Radford et al. (2019) — [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Hu et al. (2021) — [LoRA: Low-Rank Adaptation of LLMs](https://arxiv.org/abs/2106.09685)

**Docs:** [Hugging Face — Causal LM fine-tuning tutorial](https://huggingface.co/docs/transformers/tasks/language_modeling)


## 📝 Summary

In **02 — Fine-tuning Language Models** you ran the full transfer-learning recipe on **two real corpora**: pretrained a character-level language model on 593 real Usenet `sci.space` posts, measured its poor fit on 6,000 real 911 dispatch titles, fine-tuned briefly at a 10× lower learning rate, and verified with printed numbers what happened.

The measured result:

| next-char cross-entropy | before fine-tuning | after fine-tuning |
|---|---|---|
| target domain (911 titles) | 2.721 | **0.421** |
| base corpus (Usenet prose) | **1.433** | 2.011 |

The domain loss fell by 85%, and the base-corpus loss rose by 40% — **catastrophic forgetting**, measured rather than asserted. The generations show the same story: before fine-tuning the model continued the seed with space-Usenet vocabulary (`...a stound allor dod, the, eloence launol...`); afterwards it produced dispatch-shaped strings (`ems cardiac emergency . ems medical emergency ...`).

Note also the honest baseline that real data gives you: pretraining reached 1.433, not ~0. A model that memorises an invented corpus to near-zero loss makes every before/after comparison meaningless.

The Hugging Face `Trainer` reference shows the identical workflow used on real LLMs. The Unit 5 exercise reuses exactly this fine-tuning pattern.
